# Loss di training STGAN su ERA5

Per default questo notebook mostra le loss di generatore e discriminatore della CNN a 1 step gia' terminata. Legge `training_history.csv` alla fine di ogni epoca e, se disponibile, il log con le medie degli intervalli di batch. Imposta `SHOW_GAT=True` per aggiungere la GAT a 16 step anche mentre e' in esecuzione; riesegui le celle per aggiornare i grafici.

**Validation:** il protocollo di queste run addestra su tutti gli anni 1980-2004 e non registra una loss su dati tenuti fuori dal training. I grafici mostrano soltanto training; il test 2005+ non viene usato come validation. Le loss CNN 1 e GAT 16 descrivono la dinamica di addestramento, non una misura comparativa della qualita' delle anomalie: cambiano sia l'encoder sia la finestra recente.

In [ ]:
from pathlib import Path
import json, re
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'physiq_pv').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Apri il notebook dalla directory del repository physiq_pv')

SHOW_GAT = False  # La CNN completata si puo' analizzare mentre la GAT continua il training
RUNS = [
    {'label': 'CNN, 1 step', 'run_dir': 'outputs/era5_cnn_mc_paper_run2',
     'log': 'train_cnn_mc_paper_run2.log'},
]
if SHOW_GAT:
    RUNS.append({'label': 'GAT, 16 step',
                 'run_dir': 'outputs/era5_gat_mc_paper_recent16_run1',
                 'log': 'train_gat_mc_paper_recent16_run1.log'})
SAVE_FIGURES = False  # True salva PNG e PDF in outputs/era5_loss_plots
FIGURE_DIR = ROOT / 'outputs/era5_loss_plots'
print('Repository:', ROOT)

In [ ]:
PROGRESS = re.compile(r'\[stgan\] epoch=(\d+)/(\d+) batch=(\d+)/(\d+) D_mean=([\d.eE+-]+) G_mean=([\d.eE+-]+)')
LOSS_COLUMNS = ('generator_loss', 'discriminator_loss')

def read_progress(path):
    rows = []
    if path.is_file():
        with path.open(encoding='utf-8', errors='replace') as stream:
            for line in stream:
                match = PROGRESS.search(line)
                if match:
                    epoch, _, batch, total, d_loss, g_loss = match.groups()
                    rows.append({'epoch_position': int(epoch) - 1 + int(batch) / int(total),
                                 'generator_loss': float(g_loss), 'discriminator_loss': float(d_loss)})
    return pd.DataFrame(rows)

def load_run(config):
    run_dir = ROOT / config['run_dir']
    log_path = ROOT / config['log'] if config.get('log') else None
    history_path = run_dir / 'training_history.csv'
    history = pd.read_csv(history_path) if history_path.is_file() else pd.DataFrame()
    if not history.empty and not {'epoch', *LOSS_COLUMNS}.issubset(history.columns):
        raise ValueError(f'Colonne mancanti in {history_path}')
    progress = read_progress(log_path) if log_path else pd.DataFrame()
    metadata_path = run_dir / 'metadata.json'
    status = json.loads(metadata_path.read_text(encoding='utf-8')).get('status') if metadata_path.is_file() else 'in corso o incompleta'
    return {'label': config['label'], 'history': history, 'progress': progress,
            'run_dir': run_dir, 'log_path': log_path, 'status': status}

loaded = [load_run(config) for config in RUNS]
display(pd.DataFrame([{'run': run['label'], 'stato': run['status'],
                       'epoche complete': len(run['history']),
                       'punti intermedi': len(run['progress']),
                       'log trovato': bool(run['log_path'] and run['log_path'].is_file()),
                       'validation': 'non registrata'} for run in loaded]))
for run in loaded:
    if run['history'].empty and run['progress'].empty:
        print(f"Nessuna loss disponibile per {run['label']}: {run['run_dir']} / {run['log_path']}")

In [ ]:
def draw_loss(axis, run, column, color=None):
    progress, history = run['progress'], run['history']
    if not progress.empty:
        line, = axis.plot(progress['epoch_position'], progress[column], color=color,
                          linewidth=1.5, label=run['label'])
        if not history.empty:
            axis.scatter(history['epoch'], history[column], color=line.get_color(),
                         marker='o', s=26, zorder=3)
    elif not history.empty:
        axis.plot(history['epoch'], history[column], color=color, marker='o', label=run['label'])
    axis.set_xlabel('Epoca')
    axis.set_ylabel('Loss media di training')
    axis.grid(alpha=0.25)

def show_figure(figure, name):
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        figure.savefig(FIGURE_DIR / f'{name}.png', dpi=180, bbox_inches='tight')
        figure.savefig(FIGURE_DIR / f'{name}.pdf', bbox_inches='tight')
        print('Salvati:', FIGURE_DIR / f'{name}.png', 'e PDF')
    display(figure)
    plt.close(figure)

for run in loaded:
    if run['history'].empty and run['progress'].empty:
        continue
    figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for axis, column, title in zip(axes, LOSS_COLUMNS, ('Generatore: 500 x MSE + BCE', 'Discriminatore: BCE')):
        draw_loss(axis, run, column)
        axis.set_title(title)
    figure.suptitle(f"Loss di training — {run['label']}")
    show_figure(figure, 'loss_' + run['label'].lower().replace(', ', '_').replace(' ', '_'))
print('Linee: medie degli intervalli nel log. Cerchi: medie delle epoche complete.')

## Confronto visivo

Il confronto appare solo con `SHOW_GAT=True`. CNN 1 e GAT 16 cambiano contemporaneamente encoder e contesto recente: le rispettive loss non misurano direttamente la qualita' della rilevazione delle anomalie e non isolano l'effetto di un solo fattore.

In [ ]:
available = [run for run in loaded if not run['history'].empty or not run['progress'].empty]
if len(available) > 1:
    figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for index, run in enumerate(available):
        for axis, column in zip(axes, LOSS_COLUMNS):
            draw_loss(axis, run, column, color=colors[index % len(colors)])
    axes[0].set_title('Generatore: 500 x MSE + BCE')
    axes[1].set_title('Discriminatore: BCE')
    axes[0].legend()
    figure.suptitle('ERA5 STGAN — sole loss di training')
    show_figure(figure, 'cnn1_gat16_training_comparison')
elif available:
    print('Solo CNN selezionata: le sue curve sono mostrate nella sezione precedente.')
else:
    print('Nessuna loss ancora disponibile: verifica i percorsi e riesegui le celle dopo il primo log.')